In [13]:
!pip install opendatasets


In [14]:
import opendatasets as od
od.download("https://www.kaggle.com/rmisra/news-headlines-dataset-for-sarcasm-detection")

Skipping, found downloaded files in "./news-headlines-dataset-for-sarcasm-detection" (use force=True to force download)


# Import Data

In [15]:
# Import libraries
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# NLP
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('wordnet', quiet=True)

# Sklearn
from sklearn.model_selection import train_test_split

# TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Bidirectional
from tensorflow.keras.layers import GlobalMaxPool1D, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping



In [16]:
import pandas as pd

df = pd.read_json('/content/news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset.json', lines=True)

print(df.shape)
print(df.columns)

(26709, 3)
Index(['article_link', 'headline', 'is_sarcastic'], dtype='object')


In [17]:
df.head(3)

,article_link,headline,is_sarcastic
0,https://www.huffingtonpost.com/entry/versace-b...,former versace store clerk sues over secret 'b...,0
1,https://www.huffingtonpost.com/entry/roseanne-...,the 'roseanne' revival catches up to our thorn...,0
2,https://local.theonion.com/mom-starting-to-fea...,mom starting to fear son's web series closest ...,1


# EDA

In [18]:
df['is_sarcastic'].value_counts()

,count
is_sarcastic,
0,14985
1,11724


Balanced → No class weighting needed.

In [19]:
df['len'] = df['headline'].apply(lambda x: len(x.split()))
df.groupby('is_sarcastic')['len'].mean()

,len
is_sarcastic,
0,9.815616
1,9.884425


# Clean Text

In [20]:
import re
from nltk.stem import WordNetLemmatizer

lem  = WordNetLemmatizer()

def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [lem.lemmatize(w,'v') for w in words]
    return ' '.join(words)

df['clean_headline'] = df['headline'].apply(clean)

No stopwords removed — headlines are short; every word matters.

No extra NLP — we want generalization

# Train/Test Split

In [21]:
from sklearn.model_selection import train_test_split

X = df['clean_headline']
y = df['is_sarcastic']

X_train,X_test,y_train,y_test = train_test_split(
    X,y ,test_size = 0.2,
    stratify = y,random_state = 42
)

Why stratify? Ensures 50-50 split in both train and test.

# Tokenize & Pad (Text → Numbers)

In [22]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

VOCAB_SIZE = 20000
MAX_LEN = 60

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad to fixed length
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

# Define Callbacks


In [23]:
rlrp = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=2, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)

callbacks = [rlrp, early_stop]

# Build Models

In [24]:
results = {}
EPOCHS = 10
BATCH_SIZE = 32
EMBED_SIZE = 64

In [25]:
model_rnn = Sequential([
    Embedding(VOCAB_SIZE,EMBED_SIZE,input_length = MAX_LEN),
    SimpleRNN(32,dropout=0.2,recurrent_dropout= 0.2),
    Dense(1,activation = 'sigmoid')
])

model_rnn.compile(loss = 'binary_crossentropy',optimizer = 'adam',metrics = ['accuracy'])
model_rnn.summary()

history_rnn = model_rnn.fit(
    X_train_pad, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test_pad, y_test),
    callbacks=callbacks,
    verbose=1
)

loss, acc = model_rnn.evaluate(X_test_pad, y_test, verbose=0)
print(f"\nSimple RNN - Test Accuracy: {acc:.4f}\n")
results['RNN'] = acc

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 18s 13ms/step - accuracy: 0.5271 - loss: 0.7071 - val_accuracy: 0.5610 - val_loss: 0.6857 - learning_rate: 0.0010
Epoch 2/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.5435 - loss: 0.6911 - val_accuracy: 0.5610 - val_loss: 0.6860 - learning_rate: 0.0010
Epoch 3/10
661/668 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5477 - loss: 0.6902
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.00010000000474974513.
668/668 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.5478 - loss: 0.6902 - val_accuracy: 0.5610 - val_loss: 0.6869 - learning_rate: 0.0010
Epoch 4/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.5606 - loss: 0.6864 - val_accuracy: 0.5610 - val_loss: 0.6866 - learning_rate: 1.0000e-04
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.

Simple RNN - Test Accuracy: 0.5610



In [30]:
model_lstm = Sequential([
    Embedding(VOCAB_SIZE, EMBED_SIZE, input_length=MAX_LEN),
    LSTM(32, dropout=0.2, recurrent_dropout=0.2),
    Dense(1, activation='sigmoid')
])

model_lstm.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model_lstm.summary()

history_lstm = model_lstm.fit(
    X_train_pad, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test_pad, y_test),
    callbacks=callbacks,
    verbose=1
)

loss, acc = model_lstm.evaluate(X_test_pad, y_test, verbose=0)
print(f"\nLSTM - Test Accuracy: {acc:.4f}\n")
results['LSTM'] = acc

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 145s 211ms/step - accuracy: 0.5622 - loss: 0.6869 - val_accuracy: 0.5610 - val_loss: 0.6893 - learning_rate: 0.0010
Epoch 2/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 143s 212ms/step - accuracy: 0.5610 - loss: 0.6869 - val_accuracy: 0.5610 - val_loss: 0.6860 - learning_rate: 0.0010
Epoch 3/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 148s 221ms/step - accuracy: 0.5677 - loss: 0.6844 - val_accuracy: 0.5610 - val_loss: 0.6857 - learning_rate: 0.0010
Epoch 4/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 202s 221ms/step - accuracy: 0.5636 - loss: 0.6854 - val_accuracy: 0.5610 - val_loss: 0.6857 - learning_rate: 0.0010
Epoch 5/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step - accuracy: 0.5664 - loss: 0.6845
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.00010000000474974513.
668/668 ━━━━━━━━━━━━━━━━━━━━ 201s 219ms/step - accuracy: 0.5664 - loss: 0.6845 - val_accuracy: 0.5610 - val_loss: 0.6857 - learning_rate: 0.0010
Epoch 6/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 203s 221ms/step - accuracy

In [27]:
model_gru = Sequential([
    Embedding(VOCAB_SIZE, EMBED_SIZE, input_length=MAX_LEN),
    GRU(32, dropout=0.2, recurrent_dropout=0.2),
    Dense(1, activation='sigmoid')
])

model_gru.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model_gru.summary()

history_gru = model_gru.fit(
    X_train_pad, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test_pad, y_test),
    callbacks=callbacks,
    verbose=1
)

loss, acc = model_gru.evaluate(X_test_pad, y_test, verbose=0)
print(f"\nGRU - Test Accuracy: {acc:.4f}\n")
results['GRU'] = acc

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 138s 201ms/step - accuracy: 0.5558 - loss: 0.6870 - val_accuracy: 0.5610 - val_loss: 0.6862 - learning_rate: 0.0010
Epoch 2/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 134s 201ms/step - accuracy: 0.5598 - loss: 0.6865 - val_accuracy: 0.5610 - val_loss: 0.6862 - learning_rate: 0.0010
Epoch 3/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 133s 199ms/step - accuracy: 0.5631 - loss: 0.6854 - val_accuracy: 0.5610 - val_loss: 0.6857 - learning_rate: 0.0010
Epoch 4/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 138s 193ms/step - accuracy: 0.5597 - loss: 0.6862 - val_accuracy: 0.5610 - val_loss: 0.6860 - learning_rate: 0.0010
Epoch 5/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step - accuracy: 0.5585 - loss: 0.6865
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.00010000000474974513.
668/668 ━━━━━━━━━━━━━━━━━━━━ 146s 198ms/step - accuracy: 0.5585 - loss: 0.6865 - val_accuracy: 0.5610 - val_loss: 0.6863 - learning_rate: 0.0010
Epoch 6/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 142s 198ms/step - accuracy

In [26]:
model_stacked = Sequential([
    Embedding(VOCAB_SIZE, EMBED_SIZE, input_length=MAX_LEN),
    LSTM(32, dropout=0.2, recurrent_dropout=0.2, return_sequences=True),
    LSTM(32, dropout=0.2, recurrent_dropout=0.2),
    Dense(1, activation='sigmoid')
])

model_stacked.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model_stacked.summary()

history_stacked = model_stacked.fit(
    X_train_pad, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test_pad, y_test),
    callbacks=callbacks,
    verbose=1
)

loss, acc = model_stacked.evaluate(X_test_pad, y_test, verbose=0)
print(f"\nStacked LSTM - Test Accuracy: {acc:.4f}\n")
results['Stacked LSTM'] = acc

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 285s 409ms/step - accuracy: 0.5623 - loss: 0.6868 - val_accuracy: 0.5610 - val_loss: 0.6900 - learning_rate: 0.0010
Epoch 2/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 273s 408ms/step - accuracy: 0.5626 - loss: 0.6861 - val_accuracy: 0.5610 - val_loss: 0.6858 - learning_rate: 0.0010
Epoch 3/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 337s 431ms/step - accuracy: 0.5631 - loss: 0.6856 - val_accuracy: 0.5610 - val_loss: 0.6862 - learning_rate: 0.0010
Epoch 4/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 0s 389ms/step - accuracy: 0.5626 - loss: 0.6857
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.00010000000474974513.
668/668 ━━━━━━━━━━━━━━━━━━━━ 315s 420ms/step - accuracy: 0.5625 - loss: 0.6857 - val_accuracy: 0.5610 - val_loss: 0.6858 - learning_rate: 0.0010
Epoch 5/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 324s 423ms/step - accuracy: 0.5621 - loss: 0.6855 - val_accuracy: 0.5610 - val_loss: 0.6857 - learning_rate: 1.0000e-04
Epoch 6/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 313s 409ms/step - accu

In [28]:
model_bi_lstm = Sequential([
    Embedding(VOCAB_SIZE, EMBED_SIZE, input_length=MAX_LEN),
    Bidirectional(LSTM(32, dropout=0.2, recurrent_dropout=0.2, return_sequences=True)),
    GlobalMaxPool1D(),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model_bi_lstm.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model_bi_lstm.summary()

history_bi_lstm = model_bi_lstm.fit(
    X_train_pad, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test_pad, y_test),
    callbacks=callbacks,
    verbose=1
)

loss, acc = model_bi_lstm.evaluate(X_test_pad, y_test, verbose=0)
print(f"\nBidirectional LSTM - Test Accuracy: {acc:.4f}\n")
results['Bidirectional LSTM'] = acc

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 276s 403ms/step - accuracy: 0.7078 - loss: 0.5356 - val_accuracy: 0.8437 - val_loss: 0.3460 - learning_rate: 0.0010
Epoch 2/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 318s 397ms/step - accuracy: 0.9132 - loss: 0.2253 - val_accuracy: 0.8538 - val_loss: 0.3391 - learning_rate: 0.0010
Epoch 3/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 315s 386ms/step - accuracy: 0.9585 - loss: 0.1213 - val_accuracy: 0.8491 - val_loss: 0.4206 - learning_rate: 0.0010
Epoch 4/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 0s 370ms/step - accuracy: 0.9756 - loss: 0.0706
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.00010000000474974513.
668/668 ━━━━━━━━━━━━━━━━━━━━ 272s 401ms/step - accuracy: 0.9756 - loss: 0.0707 - val_accuracy: 0.8474 - val_loss: 0.4833 - learning_rate: 0.0010
Epoch 5/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 268s 401ms/step - accuracy: 0.9912 - loss: 0.0346 - val_accuracy: 0.8467 - val_loss: 0.5574 - learning_rate: 1.0000e-04
Epoch 5: early stopping
Restoring model weights from the end o

In [29]:
model_bi_gru = Sequential([
    Embedding(VOCAB_SIZE, EMBED_SIZE, input_length=MAX_LEN),
    Bidirectional(GRU(32, dropout=0.2, recurrent_dropout=0.2, return_sequences=True)),
    GlobalMaxPool1D(),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model_bi_gru.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model_bi_gru.summary()

history_bi_gru = model_bi_gru.fit(
    X_train_pad, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test_pad, y_test),
    callbacks=callbacks,
    verbose=1
)

loss, acc = model_bi_gru.evaluate(X_test_pad, y_test, verbose=0)
print(f"\nBidirectional GRU - Test Accuracy: {acc:.4f}\n")
results['Bidirectional GRU'] = acc

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 250s 364ms/step - accuracy: 0.7164 - loss: 0.5278 - val_accuracy: 0.8474 - val_loss: 0.3457 - learning_rate: 0.0010
Epoch 2/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 266s 371ms/step - accuracy: 0.9143 - loss: 0.2242 - val_accuracy: 0.8517 - val_loss: 0.3640 - learning_rate: 0.0010
Epoch 3/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 0s 337ms/step - accuracy: 0.9585 - loss: 0.1195
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.00010000000474974513.
668/668 ━━━━━━━━━━━━━━━━━━━━ 260s 368ms/step - accuracy: 0.9585 - loss: 0.1195 - val_accuracy: 0.8439 - val_loss: 0.4080 - learning_rate: 0.0010
Epoch 4/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 252s 353ms/step - accuracy: 0.9820 - loss: 0.0626 - val_accuracy: 0.8465 - val_loss: 0.4770 - learning_rate: 1.0000e-04
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.

Bidirectional GRU - Test Accuracy: 0.8474



In [31]:
for model_name,accuracy in sorted(results.items(),key = lambda x: x[1],reverse = True):
    print(f"{model_name:20}: accuracy:.4f")

Bidirectional LSTM  : accuracy:.4f
Bidirectional GRU   : accuracy:.4f
RNN                 : accuracy:.4f
Stacked LSTM        : accuracy:.4f
GRU                 : accuracy:.4f
LSTM                : accuracy:.4f


In [43]:
def predict_sarcasm(text):
    cleaned = clean(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post')
    pred = model_bi_lstm.predict(padded, verbose=0)[0][0]
    print(f"Text: {text}")
    print(f"Sarcasm Score: {pred:.4f} ({'Sarcastic' if pred > 0.5 else 'Not Sarcastic'})")

predict_sarcasm("Richard Branson pledges 3 billion to fight climate change almost as much as he spent on failed balloon trips")

Text: Richard Branson pledges 3 billion to fight climate change almost as much as he spent on failed balloon trips
Sarcasm Score: 0.8720 (Sarcastic)


In [53]:
predict_sarcasm("gourmet gifts for the foodie 2014")

Text: gourmet gifts for the foodie 2014
Sarcasm Score: 0.0170 (Not Sarcastic)


In [51]:
df[['headline','is_sarcastic']].tail(5)

,headline,is_sarcastic
26704,american politics in moral free-fall,0
26705,america's best 20 hikes,0
26706,reparations and obama,0
26707,israeli ban targeting boycott supporters raise...,0
26708,gourmet gifts for the foodie 2014,0
